# Function 3 · Week 10 decision review

Near-zero scores and controlled coordinate tests

[Detailed observations](official/exploration-review.ipynb) · [Detailed fits](official/model_predictions.ipynb) · [Previous pooled study](full-cohort/model_predictions.ipynb) · [2024 comparison](../../../side-quests/historical-cohort-2024/README.md) · [Course data-use review](../../../docs/capstone-data-policy.md)

**Decision status.** Week 10 is prepared and unsubmitted. Round 9 is the latest official return. Start with the [prepared comparison](#prepared-week-10-decision) when choosing between the two portal batches.

## Expectations

Round 9 tested x2 = 0.58 between two strong readings while keeping x1 and x3 fixed. The archive replay evaluates the saved GP and neural mean at the midpoint.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

from IPython.display import display
import numpy as np
import pandas as pd

FUNCTION = 3
WEEK_DIR = Path.cwd().resolve()
WEEK = int(WEEK_DIR.name.removeprefix("Week_"))
STAGE = WEEK_DIR.parents[2]
SHARED = STAGE / "notebooks/Shared"
SPECIAL = STAGE / "notebooks/Function_1/FunctionSpecificShared"
sys.path[:0] = [str(SHARED), str(SPECIAL)]
from weekly_peak_review import peak_review, portal_point
from query_budget import remaining_queries

analysis = WEEK_DIR / "official" if FUNCTION <= 4 else WEEK_DIR
official_path = analysis / "observations.csv"
model_snapshot = json.loads((analysis / "model-review.json").read_bytes())
observation_snapshot = json.loads((analysis / "observation-review.json").read_bytes())
feedback = json.loads((STAGE / f"submissions/round-{WEEK-1:02d}-results.json").read_bytes())
record = feedback["results"][FUNCTION - 1]
forecast = observation_snapshot["historical_forecast"]
if (model_snapshot["week"], model_snapshot["function"], model_snapshot["snapshot_sha256"]) != (
        WEEK, FUNCTION, sha256(official_path.read_bytes()).hexdigest()):
    raise ValueError("The official model review and observations describe different snapshots")
if (feedback["round"], record["function"], record["portal_input"], record["output"]) != (
        WEEK - 1, FUNCTION, forecast["portal_input"], forecast["actual_output"]):
    raise ValueError("The saved forecast must match the returned observation")
official = pd.read_csv(official_path, float_precision="round_trip")
cols = [c for c in official if c.startswith("x")]
x = official[cols].to_numpy(float)
raw_y = official.y.to_numpy(float)
later = official.source_round.to_numpy() > 0
best_row = official.loc[official.y.idxmax()]
incumbent = best_row[cols].to_numpy(float)
queries_left = remaining_queries(WEEK_DIR)
units = forecast["prediction_units"]
target = raw_y
if FUNCTION == 1:
    from function_1_model_helpers import signed_log
    target = signed_log(raw_y, scale=forecast["height_scale"])

display(pd.Series({
    "Archived forecast source": forecast["source"],
    "Submitted input": forecast["portal_input"],
    "Forecast units": units,
    "Saved GP mean": forecast["gp_mean"],
    "GP latent SD": forecast["gp_latent_sd"],
    "Saved neural mean": forecast["neural_mean"],
    "Neural seed SD": forecast["neural_seed_sd"],
}, name="Forecast frozen before the return"))

Archived forecast source    notebooks/Function_3/Week_09/official/figures/...
Submitted input                                    0.520000-0.580000-0.450000
Forecast units                                                       output y
Saved GP mean                                                       -0.014328
GP latent SD                                                         0.017962
Saved neural mean                                                   -0.016313
Neural seed SD                                                       0.000179
Name: Forecast frozen before the return, dtype: object

## Results

The midpoint scored below both endpoints. Aby’s one-coordinate tests gave a clearer local comparison than the earlier broad search. The 2024 archive has additional near-zero readings in the same general region.

In [2]:
latest_fitted = forecast["actual_in_prediction_units"]
display(pd.Series({"Returned score": record["output"], "Best before return": record["best_before"],
                   "Best now": record["best_after"], "Return rank": observation_snapshot["latest_rank"],
                   "Actual in forecast units": latest_fitted,
                   "Actual minus GP mean": forecast["actual_minus_gp_mean"],
                   "Actual minus neural mean": forecast["actual_minus_neural_mean"]}, name="Official result"))

# These references describe observed outcomes. The decision fits use official rows.
def optional_reference(folder, file, audit_name, expected):
    path = folder / file
    manifest = folder / audit_name
    if not (path.is_file() and manifest.is_file()):
        return None
    manifest_data = json.loads(manifest.read_bytes())
    if sha256(path.read_bytes()).hexdigest() != expected(manifest_data):
        raise ValueError(f"Reference hash mismatch: {path}")
    return pd.read_csv(path, float_precision="round_trip")

current_reference = optional_reference(
    STAGE / "data/cohort/2026-11-25", f"function_{FUNCTION}/observations.csv", "dataset-audit.json",
    lambda a: a["summary"][FUNCTION - 1]["sha256"],
)
archive = optional_reference(
    STAGE / "data/historical-cohort/2024", f"function_{FUNCTION}.csv", "intake-audit.json",
    lambda a: a["functions"][str(FUNCTION)]["sha256"],
)
reference_rows = []
if current_reference is not None:
    peer = current_reference.loc[current_reference.observation_id.str.startswith("peer-")]
    reference_rows.append(("Published current cohort", peer))
if archive is not None:
    reference_rows.append(("2024 archive, including extra starters", archive))
benchmarks = []
for label, frame in reference_rows:
    row = frame.loc[frame.y.idxmax()]
    benchmarks.append({"Scope": label, "Highest measured": row.y,
                       "Difference from your best": row.y - best_row.y,
                       "Distance to your best": np.linalg.norm(row[cols].to_numpy(float) - incumbent)})
if benchmarks:
    display(pd.DataFrame(benchmarks).round(8))
else:
    print("Local comparison snapshots are unavailable for this run.")

Returned score             -0.008109
Best before return         -0.004952
Best now                   -0.004952
Return rank                 2.000000
Actual in forecast units   -0.008109
Actual minus GP mean        0.006219
Actual minus neural mean    0.008204
Name: Official result, dtype: float64

,Scope,Highest measured,Difference from your best,Distance to your best
0,Published current cohort,-0.000140,0.004812,0.060000
1,"2024 archive, including extra starters",-0.000253,0.004699,0.081237


In [3]:
line = official.loc[official.x1.eq(best_row.x1) & official.x3.eq(best_row.x3),
                    ["observation_id", "x1", "x2", "x3", "y"]]
display(line.sort_values("x2"))
print(f"Distance in score to the hypothesised zero ceiling: {-float(best_row.y):.8g}")
if archive is not None:
    nearby = np.linalg.norm(archive[cols].to_numpy() - [0.52, 0.55, 0.45], axis=1) <= 0.032
    display(archive.loc[nearby, [*cols, "y"]].nlargest(4, "y"))

,observation_id,x1,x2,x3,y
23,round-09,0.52,0.58,0.45,-0.008109
22,round-08,0.52,0.61,0.45,-0.004952


Distance in score to the hypothesised zero ceiling: 0.0049520008


,x1,x2,x3,y
73,0.519622,0.530912,0.468558,-0.000253
77,0.524501,0.533461,0.469730,-0.000272
76,0.536937,0.537095,0.471463,-0.000316
74,0.517332,0.527787,0.453002,-0.002123


## Predictions

**Local peak and global evidence**

The current incumbent is near the best measured region, but the exact location of a local optimum is unresolved. A ceiling at zero is a plausible hypothesis given the course analogy and observed signs. It is not a proven bound on the implemented portal score. If zero were a verified upper bound, the incumbent would be within about 0.004952 raw-score units of the global maximum regardless of what the GP predicts.

### Prepared Week 10 decision

**Prepared and unsubmitted as of 25 November 2026.** Round 9 is the latest confirmed feedback. Four portal queries remain per function, including Week 10. Use the cohort-informed route only if the course permits that use, including the exact published F2 repeat. The official-data fallback is ready if the rules remain unclear near the portal deadline.

The revised input changes only x2, placing it 0.005 from Aby's near-zero measurement. The advisory experiment ranked a distant x2≈0.95 point first on an extrapolated positive mean. The official fallback follows the measured direction from x2=0.58 to 0.61 and tests x2=0.65 on the same x1/x3 line. Zero remains a hypothesis about the ceiling.

**Week 11 depends on the return.** Tighten x2 spacing on the controlled slice if the chosen side improves the score. A weak return favours the best measured setting and a carefully spaced check.

[Week 10 course check and exact batches](../../../submissions/Week_10/README.md) · [Cohort advice and replay](../../../submissions/Week_10/advisory-review.md)

The estimates below use the advisory review's official-data GP comparator and predictive return standard deviation, which includes modelled observation noise. F8 uses the validated starter quadratic and omits a standard deviation because that fit supplies no probabilistic interval. The rows are proposed inputs with model estimates.

In [4]:
from week_10_decision_record import decision_comparison

with pd.option_context("display.max_colwidth", None):
    display(decision_comparison(STAGE, FUNCTION, official))

,Portal input,Official GP mean (score),Predictive return SD
Stage,,,
Initial proposal,0.520000-0.555000-0.450000,-0.010775,0.001183
Advisory experiment,0.295456-0.950070-0.428581,0.026244,0.033515
Revised cohort-informed,0.520000-0.555000-0.450000,-0.010775,0.001183
Official-data fallback,0.520000-0.650000-0.450000,-0.001147,0.001653


### Conditional peak check

Two Gaussian processes with fitted noise share the same candidate set. A local candidate lies within the stated Euclidean radius of the observed incumbent. The columns report the fraction of joint latent draws in which none of the candidates in that set beats the incumbent by more than the working tolerance. The broader search includes Sobol points, corners, observed coordinates and surrogate-suggested probes. It cannot certify the continuous function or rule out unrepresented peaks. The nine grouped official return holdouts and nominal 90% observation-band counts indicate predictive fit. They do not calibrate the global-maximum event. The tolerance and radius are analysis choices.

In [5]:
radius = 0.06
tolerance = 0.004
peak = peak_review(x, target, later, radius=radius, tolerance=tolerance,
                   length_scales=[0.17, 0.17, 0.2], noise=[0.04, 5e-05, 0.4],
                   minimum_distance=model_snapshot["minimum_query_distance"])
print(f"Local radius {radius:g}. Score tolerance {tolerance:g} in {units}.")
print(f"Joint posterior over {peak['candidate_count']} candidates, including {peak['local_count']} local."
      f" {peak['probability_draws']} draws per kernel.")
display(peak["summary"].round(5))
print("Adaptive two-query comparison on the Matérn shortlist")
one_query = peak["acquisitions"]["Matérn 3/2"].nlargest(1, "One-query EI")
print(f"Best one-query input: {one_query.iloc[0]['Input']}")
display(peak["acquisitions"]["Matérn 3/2"].head(5).round(6))

Local radius 0.06. Score tolerance 0.004 in output y.
Joint posterior over 454 candidates, including 143 local. 1024 draws per kernel.


,Local set within tolerance,Broader set within tolerance,Broader gap median,Broader gap 90th pct,Return MAE,Return RMSE,In 90% observation band,Return holdouts,Fitted noise SD,Fit warnings
Model,,,,,,,,,,
Matérn 3/2,0.00488,0.00098,0.07355,0.12155,0.01085,0.01355,9,9,0.00052,3
RBF,0.00000,0.00000,0.08497,0.13405,0.01513,0.01641,9,9,0.00052,2


Adaptive two-query comparison on the Matérn shortlist
Best one-query input: 0.557312-0.996482-0.429093


,Input,Region,Mean,Observed SD,One-query EI,Two-query gain
4,0.017606-0.993400-0.420414,Elsewhere,0.020247,0.044865,0.033249,0.045894
2,0.097780-0.968332-0.424677,Elsewhere,0.022941,0.040393,0.033758,0.044862
6,0.667283-0.918471-0.419613,Elsewhere,0.021510,0.039222,0.032310,0.044624
0,0.557312-0.996482-0.429093,Elsewhere,0.027554,0.032227,0.035147,0.044602
3,0.574795-0.963664-0.417219,Elsewhere,0.022682,0.040873,0.033714,0.044284


### Strategy for the remaining queries

Use one-coordinate experiments when the global surrogate has weak local discrimination. A short local response surface over the best-supported slice can guide step size. Use noise-aware improvement if repeat evidence makes measurement variation material.

The code compares one-query expected improvement in the returned score with a two-query approximation. It conditions on a first outcome, then chooses a different point on a shortlist. Both steps apply the saved minimum-spacing rule. It holds fitted kernel parameters fixed. Refit after each actual result and reassess the next action. The four rounds below are conditional decisions.

| Round | Action given the evidence so far |
| --- | --- |
| 10 | Hold x1 and x3 fixed. The revised route tests x2=0.555 near the peer measurement. The official fallback tests x2=0.65. |
| 11 | Tighten x2 spacing on the controlled slice if the chosen side improves the score. A weak return favours the best measured setting and a carefully spaced check. |
| 12 | Hold the best-supported x2 and x3 fixed and test a single x1 shift if the remaining score gap justifies it. |
| 13 | Exploit the best-supported local setting. Treat any near-zero result as evidence about the hypothesised ceiling. |

[Frazier, Bayesian optimisation tutorial](https://arxiv.org/abs/1807.02811) · [Letham et al., noisy expected improvement](https://arxiv.org/abs/1706.07094) · [Kolda, Lewis and Torczon, direct search](https://epubs.siam.org/doi/10.1137/S003614450242889)

In [6]:
print(f"{queries_left} query opportunities per function, including Week {WEEK}.")
destination = WEEK_DIR / "weekly-peak-review.json"
summary = {"function": FUNCTION, "week": WEEK, "queries_remaining": queries_left,
           "official_snapshot_sha256": model_snapshot["snapshot_sha256"],
           "candidate_count": peak["candidate_count"], "local_count": peak["local_count"],
           "probability_draws": peak["probability_draws"], "local_radius": radius,
           "tolerance": tolerance, "target_units": units,
           "models": peak["summary"].reset_index().to_dict(orient="records"),
           "matérn_shortlist": peak["acquisitions"]["Matérn 3/2"].to_dict(orient="records")}
if FUNCTION == 8:
    summary["structural_fit"] = quadratic
destination.write_bytes((json.dumps(summary, indent=2, allow_nan=False) + "\n").encode())

4 query opportunities per function, including Week 10.


3791

### Course lifecycle

1. **Define the goal and scope** – choose the next official query while assessing how close the incumbent might be to the peak.
2. **Gather the data** – preserve the official snapshot, the frozen predictions and labelled published comparisons.
3. **Explore the data** – compare the returned result with local and external observations.
4. **Clean and preprocess the data** – verify hashes and portal alignment. Preserve the recorded raw score.
5. **Select and engineer features** – use the original coordinates, with a signed-log target for F1. Structural fits declare their terms separately.
6. **Define the machine-learning task** – regression supports maximisation and assessment of the incumbent.
7. **Partition the data** – withhold each official return and any matching portal coordinate for the predictive check.
8. **Select and train candidate methods** – compare two kernel assumptions and function-specific hypotheses.
9. **Evaluate and interpret the results** – judge forecast reliability and the finite-domain comparison before choosing how to spend the remaining budget.
10. **Deploy and iterate** – deployment is not applicable. The next portal return will update the review and the query plan.